<a href="https://colab.research.google.com/github/AHichem/bridge-deterioration-nbi/blob/main/notebooks/02_sql_model.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [3]:
!pip uninstall -y -q ipython-sql
!pip install -q jupysql

from google.colab import drive
drive.mount('/content/drive')
DATA_DIR = '/content/drive/MyDrive/nbi'

import duckdb
con = duckdb.connect(f'{DATA_DIR}/nbi.duckdb')

%load_ext sql
%sql con
%config SqlMagic.displaylimit = 20

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
The sql extension is already loaded. To reload it, use:
  %reload_ext sql


In [ ]:
%%sql
CREATE OR REPLACE TABLE inspections_raw AS
SELECT
    CAST(regexp_extract(filename, '(20\d{2})', 1) AS INTEGER) AS year,
    STATE_CODE_001, STRUCTURE_NUMBER_008, COUNTY_CODE_003, LAT_016, LONG_017,
    RECORD_TYPE_005A, OWNER_022, FUNCTIONAL_CLASS_026, YEAR_BUILT_027,
    YEAR_RECONSTRUCTED_106, SERVICE_ON_042A, STRUCTURE_KIND_043A, STRUCTURE_TYPE_043B,
    MAIN_UNIT_SPANS_045, STRUCTURE_LEN_MT_049, DECK_WIDTH_MT_052,
    ADT_029, PERCENT_ADT_TRUCK_109,
    DECK_COND_058, SUPERSTRUCTURE_COND_059, SUBSTRUCTURE_COND_060, CULVERT_COND_062,
    BRIDGE_CONDITION,
    DATE_OF_INSPECT_090, INSPECT_FREQ_MONTHS_091
FROM read_csv(
    [   -- 2016 à 2025 : nbi_2015.txt n'est pas un vrai fichier NBI (125 lignes, aucune colonne attendue)
        '/content/drive/MyDrive/nbi/raw/nbi_201[6-9].txt',
        '/content/drive/MyDrive/nbi/raw/nbi_202[0-9].txt'
    ],
    encoding = 'latin-1',    -- les fichiers NBI ne sont pas en UTF-8 (accents)
    delim = ',',
    quote = '''',            -- le qualificateur de texte est une apostrophe
    header = true,
    all_varchar = true,      -- tout en texte pour l'instant
    union_by_name = true,    -- aligne les colonnes par nom si l'ordre change selon l'année
    filename = true          -- ajoute le chemin du fichier, d'où on extrait l'année
);

In [ ]:
%%sql
SELECT year, COUNT(*) AS n_rows
FROM inspections_raw
GROUP BY year
ORDER BY year;